<a href="https://colab.research.google.com/github/ravikanthd1981/rav4/blob/master/Full_Code_NLP_RAG_Ravikanth_Doddapaneni_V12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [3]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
# NOTE: --no-deps stops pip from reinstalling numpy here. Reinstalling numpy is what
# broke Colab's pandas/scipy earlier ("numpy dtype size changed" / "_center").
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir --no-deps -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir --no-deps -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 88.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
# For installing the libraries & downloading models from HF Hub
# STRATEGY: install ONLY the higher-level libraries Colab does not already have, and use
# --no-deps for sentence-transformers so it does NOT upgrade torch. We deliberately do
# NOT install numpy / pandas / scipy / scikit-learn / torch / torchvision / transformers,
# leaving Colab's mutually-consistent stack intact. This avoids BOTH the
# "numpy _center" and the "torchvision::nms does not exist" errors.
!pip install -q --no-deps sentence-transformers==5.1.1

# diskcache + typing_extensions are llama-cpp-python's pure-Python deps (skipped by the
# --no-deps install in the llama-cpp cell above). They do NOT depend on numpy, so it is
# safe to install them here. Fixes "No module named diskcache".
!pip install -q langchain==0.3.27 langchain-community==0.3.31 pymupdf==1.26.5 tiktoken==0.12.0 chromadb==1.1.1 diskcache

# Verify the key imports in FRESH subprocesses. All should print OK with NO traceback.
!python -c "import numpy, torch, torchvision; print('OK numpy', numpy.__version__, '| torch', torch.__version__, '| torchvision', torchvision.__version__)"
!python -c "import sentence_transformers, transformers; print('OK sentence-transformers', sentence_transformers.__version__, '| transformers', transformers.__version__)"
!python -c "from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings; print('OK langchain SentenceTransformerEmbeddings import')"
!python -c "from llama_cpp import Llama; print('OK llama_cpp import')"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 k

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd
from google.colab import drive
#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [4]:
# Quick environment check -> everything should import cleanly. RUN THIS ONLY AFTER
# Runtime > Restart session. If it fails, do Runtime > Disconnect and delete runtime
# and start again from the install cells (do NOT re-run install cells after restarting).
import numpy, scipy, sklearn, pandas, torch, torchvision, sentence_transformers
print("numpy                :", numpy.__version__)
print("pandas               :", pandas.__version__)
print("scipy                :", scipy.__version__)
print("scikit-learn         :", sklearn.__version__)
print("torch                :", torch.__version__)
print("torchvision          :", torchvision.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

numpy                : 2.0.2
pandas               : 2.2.2
scipy                : 1.16.3
scikit-learn         : 1.6.1
torch                : 2.11.0+cu128
torchvision          : 0.26.0+cu128
sentence-transformers: 5.1.1


## Question Answering using LLM

#### Downloading and Loading the model

In [2]:
# Model repository and file on the Hugging Face Hub
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

# Download the model weights from the HF Hub
model_path = hf_hub_download(
    repo_id=model_name_or_path,   # repo id
    filename=model_basename       # model file name
)

# Load the model (T4-GPU runtime).
# If running on CPU only, reduce n_ctx and remove n_gpu_layers / n_batch.
llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=36,
    n_batch=512
)

mistral-7b-instruct-v0.2.Q6_K.gguf: reconstructing file:   0%|          |  0.00B / 5.94GB            

mistral-7b-instruct-v0.2.Q6_K.gguf: downloading bytes:           |  0.00B            

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [5]:
def response(query,max_tokens=256,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [6]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input_1)

'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs: Ensure airway patency, adequate breathing, and circulatory support. Provide high-flow oxygen via a non-rebreather mask or endotracheal tube if necessary. Initiate intravenous fluids to maintain adequate blood pressure and organ perfusion.\n3. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on the suspected source of infection and local microbiology data. Consider obtaining cultures before administering antibiotics if clin

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [7]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that worsens over time.\n2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea.\n3. Nausea and vomiting: Vomiting is a common symptom of appendicitis, especially in the early stages.\n4. Fever: A fever may be present, particularly if the appendix has ruptured or perforated.\n5. Constipation or diarrhea: Some people with appendicitis experience constipation, while others have diarrhea.\n6. Abdominal swelling: The abdomen may become swollen and tender to the touch.\n7. Inability to pass gas: Passing gas can be difficult due to the'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [8]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_3)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications.\n\nThere are several treatments that have been shown to be effective in addressing sudden patchy hair loss:\n\n1. Corticosteroids: These are anti-inflammatory drugs that can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally, depending on the severity of the condition.\n2. Minoxidil: This is a medication that has been shown to promote hair growth in some people with alopecia areata. It works by incre

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [9]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4)

Llama.generate: prefix-match hit


'\n\nA person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:\n\n1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.\n2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a brain injury, such as pain, swelling, seizures, or infections.\n3. Surgery: In some cases, surgery may be necessary to remove blood clots, repair skull fractures, or relieve pressure on the brain.\n4. Rehabilitation: Rehabilitation is an essential component of treatment for brain injuries. It may include physical therapy, occupational therapy, speech and langua

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [10]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_5)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sling, or other available materials to immobilize the leg and prevent movement. Be sure not to apply too much pressure on the injury site.\n4. Provide pain relief: Offer over-the-counter pain medication, such as acetaminophen or ibuprofen, to help manage pain.\n5. Seek medical attention: If the fracture is severe or if you suspect that there may be other injuries, seek medical help as soon as possible.\n\nOnce you've ensured the person's safety and stabi

#### Observations - Question Answering using the base LLM

- The base Mistral-7B model answers only from its pre-trained knowledge, with **no access to the Merck Manual**. Answers are fluent but tend to be **generic**, and for specifics such as drug trade names, dosages, or step-by-step critical-care protocols the model can be **incomplete or occasionally inaccurate (hallucinate)**.
- With `temperature=0` the answers are deterministic and repeatable, which is desirable for a clinical assistant.
- `max_tokens` is set to 256 so the longer answers (e.g., the sepsis protocol) can complete instead of being cut off; this was raised from the default of 128 after seeing early answers truncate.

_Run the cells on a T4-GPU and refine these bullets to match the exact answers you obtain._

## Question Answering using LLM with Prompt Engineering

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [11]:
# System prompt to steer the model towards concise, reliable medical answers
system_prompt = (
    "You are a knowledgeable medical assistant. "
    "Provide accurate, concise, and clinically relevant answers to the user's medical question. "
    "If the question involves emergency care, outline the key steps clearly. "
    "Do not fabricate information."
)

# Combination 1 -> temperature=0.0, top_p=0.95, top_k=50, max_tokens=256
user_input_1 = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"
response(user_input_1, max_tokens=256, temperature=0.0, top_p=0.95, top_k=50)

Llama.generate: prefix-match hit


'\nSepsis is a life-threatening condition caused by a severe infection that can lead to organ failure. In a critical care unit, managing sepsis involves the following steps:\n1. Early recognition and diagnosis: Identify sepsis early based on clinical signs such as fever, tachycardia, respiratory distress, and altered mental status. Use laboratory tests like blood cultures, lactate levels, and complete blood count to confirm the diagnosis.\n2. Immediate fluid resuscitation: Administer intravenous fluids to maintain adequate blood pressure and organ perfusion. The goal is to administer 30 mL/kg of crystalloid fluid over the first three hours.\n3. Antibiotics administration: Start broad-spectrum antibiotics as soon as possible based on the suspected infection site and microbiological culture results.\n4. Vasopressor support: If sepsis leads to hypotension that is not responsive to fluid resuscitation, administer vasopressors to maintain adequate blood pressure.\n5. Corticosteroids: Consid

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [12]:
# Combination 2 -> temperature=0.3, top_p=0.90, top_k=50, max_tokens=256
user_input_2 = system_prompt + "\n" + "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2, max_tokens=256, temperature=0.3, top_p=0.90, top_k=50)

Llama.generate: prefix-match hit


"\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch located in the lower right side of the abdomen. Common symptoms include:\n1. Sudden and persistent pain in the lower right abdomen that may migrate to the middle abdomen or back.\n2. Loss of appetite and feeling sick to your stomach (nausea).\n3. Vomiting.\n4. Fever, which may be low-grade at first but can rise as high as 103 degrees Fahrenheit (39.4 degrees Celsius) as the condition worsens.\n5. Constipation or diarrhea.\n6. Abdominal swelling and tenderness.\n7. Inability to pass gas or have a bowel movement.\nAppendicitis cannot be cured with medicine alone, as the inflammation can lead to the appendix rupturing if left untreated. If the appendix ruptures, it can cause peritonitis, a serious infection of the abdominal cavity that requires immediate surgical intervention. Therefore, if you suspect appendicitis, it's essential to seek medical attention right"

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [13]:
# Combination 3 -> temperature=0.5, top_p=0.95, top_k=10, max_tokens=300
user_input_3 = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_3, max_tokens=300, temperature=0.5, top_p=0.95, top_k=10)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that causes hair loss in small, round patches on the scalp or other parts of the body. It can affect people of all ages and genders. The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system.\n\nEffective treatments for addressing sudden patchy hair loss include:\n\n1. Corticosteroids: These medications can help reduce inflammation and suppress the immune response that causes hair loss. They can be applied topically or taken orally.\n2. Immunotherapy: This treatment involves injecting small doses of certain substances into the bald spots to stimulate the immune system and promote hair regrowth.\n3. Minoxidil: This medication is applied topically and can help slow down hair loss and promote new hair growth in some people.\n4. Wigs or hairpieces: If other treatments are not effective, wearing a wig or hairpiece can help cover up the bald spots.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [14]:
# Combination 4 -> temperature=0.7, top_p=0.95, top_k=50, max_tokens=300
user_input_4 = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4, max_tokens=300, temperature=0.7, top_p=0.95, top_k=50)

Llama.generate: prefix-match hit


'\nA person with a brain injury may require various treatments depending on the severity and location of the injury. Here are some common treatments:\n1. Emergency care: If the injury is severe, the person may need emergency treatment to manage symptoms such as breathing difficulties, seizures, or increased intracranial pressure. This may include medications, surgery, or intubation.\n2. Rehabilitation: Rehabilitation is an essential part of treating a brain injury. It can help improve function and reduce disabilities. Rehabilitation may include physical therapy, occupational therapy, speech therapy, and cognitive rehabilitation.\n3. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as seizures, pain, or depression.\n4. Surgery: In some cases, surgery may be necessary to remove clots, repair skull fractures, or relieve pressure on the brain.\n5. Assistive devices: People with brain injuries may require assistive devices such as wheelchairs, 

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [15]:
# Combination 5 -> a more detailed system prompt + temperature=0.2, max_tokens=350
detailed_system_prompt = (
    "You are an experienced clinical assistant. Answer the medical question in a structured way: "
    "briefly state the assessment, then list the recommended steps or treatments in order of priority, "
    "and mention important precautions. Keep it factual and avoid speculation."
)
user_input_5 = detailed_system_prompt + "\n" + "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_5, max_tokens=350, temperature=0.2, top_p=0.95, top_k=50)

Llama.generate: prefix-match hit


"\nAssessment:\n- Determine the location, severity, and openness of the fracture.\n- Assess for signs of shock: pale, clammy skin; rapid heartbeat; weak pulse; low blood pressure; anxiety or confusion.\n- Check for any associated injuries such as damage to nerves or vessels.\n- Evaluate the patient's overall condition, including their ability to bear weight and mobility.\nTreatment steps in order of priority:\n1. Immobilize the fractured leg using a splint or cast to prevent further movement and promote healing. This can be done with makeshift materials if necessary until professional help arrives.\n2. Provide adequate pain relief using over-the-counter medications like acetaminophen or ibuprofen, or prescription painkillers if needed.\n3. Monitor vital signs closely and provide fluids to prevent dehydration and maintain blood pressure.\n4. Transport the patient to a medical facility as soon as possible for further evaluation and definitive treatment.\nImportant precautions:\n- Do not 

#### Observations - Prompt Engineering & Parameter Tuning

Five prompt/parameter combinations were tried (varying the system instruction, `temperature`, `top_p`, `top_k`, and `max_tokens`):

| # | Query | temperature | top_p | top_k | max_tokens |
|---|-------|:-----------:|:-----:|:-----:|:----------:|
| 1 | Sepsis | 0.0 | 0.95 | 50 | 256 |
| 2 | Appendicitis | 0.3 | 0.90 | 50 | 256 |
| 3 | Hair loss | 0.5 | 0.95 | 10 | 300 |
| 4 | Brain injury | 0.7 | 0.95 | 50 | 300 |
| 5 | Leg fracture | 0.2 | 0.95 | 50 | 350 |

- Adding a **system prompt** that defines the assistant's role yields more **structured, clinically framed** answers than the bare queries.
- **Lower temperature (0-0.2)** gives focused, consistent answers; **higher temperature (0.5-0.7)** adds detail and alternatives but raises the risk of drift/hallucination.
- Increasing **max_tokens** lets longer protocols finish; constraining **top_k** narrows the vocabulary and makes answers more conservative.
- Prompt engineering improves the *form* of the answer, but the content is still bounded by the model's training data - which motivates **RAG** to ground answers in the manual.

_Verify against your actual outputs and note which combination worked best for each question._

## Data Preparation for RAG

### Loading the Data

In [16]:
# Google Drive mount
drive.mount('/content/drive')

# Path to the medical manual PDF (place the file in the working directory / Colab session)
manual_pdf_path = "/content/drive/MyDrive/Colab/MedicalAssistant/medical_diagnosis_manual.pdf"

# Initialize the PDF loader and load the document
pdf_loader = PyMuPDFLoader(manual_pdf_path)
manual = pdf_loader.load()

Mounted at /content/drive


### Data Overview

#### Checking the first 5 pages

In [17]:
for i in range(5):
    print(f"Page Number : {i+1}", end="\n")
    print(manual[i].page_content, end="\n")

Page Number : 1
ravidodda@gmail.com
Q7XBIO4Z6P
This file is meant for personal use by ravidodda@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
ravidodda@gmail.com
Q7XBIO4Z6P
This file is meant for personal use by ravidodda@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......................................................................................................................................

#### Checking the number of pages

In [18]:
len(manual)

4114

### Data Chunking

In [19]:
# Split the document into token-based chunks with overlap to preserve context continuity
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=128
)

document_chunks = pdf_loader.load_and_split(text_splitter)
print("Number of chunks:", len(document_chunks))
document_chunks[0].page_content

Number of chunks: 9360


'ravidodda@gmail.com\nQ7XBIO4Z6P\nThis file is meant for personal use by ravidodda@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

### Embedding

In [20]:
# Load a sentence-transformer embedding model
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Sanity check: embed two chunks and confirm the vector dimension
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

print("Dimension of the embedding vector ", len(embedding_1))
len(embedding_1) == len(embedding_2)

/tmp/ipykernel_3152/2330598441.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dimension of the embedding vector  384


True

### Vector Database

In [21]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
    os.makedirs(out_dir)

# Build the Chroma vector store from the document chunks and the embedding model
vectorstore = Chroma.from_documents(
    document_chunks,       # document chunks
    embedding_model,       # embedding model
    persist_directory=out_dir
)

# Reload the persisted store and run a quick similarity search
vectorstore = Chroma(persist_directory=out_dir, embedding_function=embedding_model)
vectorstore.similarity_search("What is the protocol for managing sepsis in a critical care unit?", k=3)

/tmp/ipykernel_3152/3601933030.py:14: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir, embedding_function=embedding_model)


[Document(metadata={'subject': '', 'moddate': '2026-08-10T04:35:02+00:00', 'modDate': 'D:20260810043502Z', 'creator': 'Atop CHM to PDF Converter', 'creationDate': 'D:20120615054440Z', 'file_path': '/content/drive/MyDrive/Colab/MedicalAssistant/medical_diagnosis_manual.pdf', 'source': '/content/drive/MyDrive/Colab/MedicalAssistant/medical_diagnosis_manual.pdf', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creationdate': '2012-06-15T05:44:40+00:00', 'total_pages': 4114, 'format': 'PDF 1.7', 'trapped': '', 'author': '', 'page': 2400, 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'keywords': ''}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurolo

### Retriever

In [22]:
# Configure the retriever to return the top-k most similar chunks
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

rel_docs = retriever.get_relevant_documents("What is the protocol for managing sepsis in a critical care unit?")
rel_docs

/tmp/ipykernel_3152/1619580787.py:7: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("What is the protocol for managing sepsis in a critical care unit?")


[Document(metadata={'subject': '', 'total_pages': 4114, 'creator': 'Atop CHM to PDF Converter', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'page': 2400, 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'keywords': '', 'format': 'PDF 1.7', 'source': '/content/drive/MyDrive/Colab/MedicalAssistant/medical_diagnosis_manual.pdf', 'creationDate': 'D:20120615054440Z', 'author': '', 'moddate': '2026-08-10T04:35:02+00:00', 'modDate': 'D:20260810043502Z', 'file_path': '/content/drive/MyDrive/Colab/MedicalAssistant/medical_diagnosis_manual.pdf', 'creationdate': '2012-06-15T05:44:40+00:00', 'trapped': ''}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurolo

### System and User Prompt Template

In [24]:
qna_system_message = (
    "You are an assistant to medical professionals. "
    "Answer the user's question using only the information provided in the context below. "
    "The context contains excerpts retrieved from a trusted medical manual. "
    "If the answer cannot be found in the context, respond with "
    "'I don't have enough information in the provided context to answer this question.' "
    "Be accurate, concise, and clinically relevant."
)

qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### Response Function

In [25]:
def generate_rag_response(user_input,k=3,max_tokens=256,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [26]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1, top_k=20)

Llama.generate: prefix-match hit


"###Answer\nThe protocol for managing sepsis in a critical care unit includes obtaining cultures of blood and any other appropriate specimens, administering empiric antibiotics after cultures are obtained, adjusting antibiotics according to culture and susceptibility testing results, surgically draining any abscesses, removing internal devices that may be the source of bacteria, and monitoring vital signs, fluid intake and output, temperature, blood pressure, pulse, respiration rate, electrolytes, CBC, Mg, phosphate, Ca levels for patients with arrhythmias, and liver enzymes and coagulation profiles for patients receiving TPN. Point-of-care testing can be used to do certain blood tests at the patient's bedside or unit. Other emerging therapies include cooling for hyperthermia and early treatment of renal failure with continuous venovenous hemofiltration."

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [27]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2)

Llama.generate: prefix-match hit


"###Answer\nThe common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; shifting pain to the right lower quadrant; increased pain with cough and motion; direct and rebound tenderness at McBurney's point; Rovsing sign (pain felt in the right lower quadrant with palpation of the left lower quadrant); psoas sign (pain felt when passively extending the right hip joint); or obturator sign (pain caused by passive internal rotation of the flexed thigh). Low-grade fever is also common. However, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur.\n\nAppendicitis cannot be cured via medicine alone. The treatment for appendicitis is surgical removal through open or laparoscopic appendectomy. If the appendix is perforated, antibiotics should be continued until the patient's temperature and WBC count have normalized or for a fixed course according to the surgeon's preference.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [28]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3)

Llama.generate: prefix-match hit


'###Answer\nThe treatment options for sudden patchy hair loss, also known as alopecia areata, include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The cause of alopecia areata is believed to be an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. It is important to note that the effectiveness of these treatments may vary, and some may have adverse effects such as decreased libido, erectile and ejaculatory dysfunction, hypersensitivity reactions, gynecomastia, and myopathy. Hormonal modulators like oral contraceptives or spironolactone may be useful for female-pattern hair loss associated with hyperandrogenemia. In severe cases, surgical options such as follicle transplant, scalp flaps, and alopecia reduction may be considered. Underlying disorders should also be treated if they are 

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [29]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4)

Llama.generate: prefix-match hit


"###Answer\nThe context suggests that there is no specific treatment for brain tissue damage beyond supportive care. This includes preventing systemic complications such as pneumonia, UTI, and thromboembolic disease; providing good nutrition; and preventing pressure ulcers. For patients whose coma exceeds 24 hours and who have persistent neurologic deficits, a prolonged period of rehabilitation is often required. This should be planned early and provided through a team approach that combines physical, occupational, and speech therapy, skill-building activities, and counseling to meet the patient's social and emotional needs. Brain injury support groups may also provide assistance to families. Late seizures can develop in a small percentage of patients, often weeks, months, or even years later. Spastic motor impairment, gait and balance disturbances, ataxia, and sensory losses may occur. A persistent vegetative state can result from a TBI that destroys forebrain cognitive functions but 

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [30]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5)

Llama.generate: prefix-match hit


"###Answer\nBased on the context provided, if a person has fractured their leg during a hiking trip, they should first receive appropriate medical attention. The leg should be immobilized immediately by splinting to prevent further injury and decrease pain. Pain is typically treated with opioids. Definitive treatment often involves reduction, which may require analgesia or sedation. Closed reduction is maintained by casting, while open reduction is maintained by various surgical hardware.\n\nAdditionally, RICE (rest, ice, compression, elevation) principles should be applied to help speed healing and minimize swelling and pain. Rest prevents further injury, ice minimizes swelling, compression reduces swelling and provides support, and elevation helps reduce swelling and improves venous return. Ice should be applied intermittently during the first 24 to 48 hours for 15 to 20 minutes as often as possible. The leg can be compressed with a splint, an elastic bandage, or a Jones compression 

### Fine-tuning

In [32]:
# Fine-tuning: try at least 5 combinations of retriever k and LLM generation parameters.
# NOTE: changing chunk_size / chunk_overlap would require rebuilding the vector store above;
# here we tune the retriever "k" together with the generation parameters.

questions = [
    "What is the protocol for managing sepsis in a critical care unit?",
    "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
]

tuning_configs = [
    {"k": 3, "max_tokens": 200, "temperature": 0.0, "top_p": 0.95, "top_k": 20},  # tight & deterministic
    {"k": 5, "max_tokens": 256, "temperature": 0.3, "top_p": 0.95, "top_k": 50},  # more context
    {"k": 2, "max_tokens": 256, "temperature": 0.5, "top_p": 0.95, "top_k": 10},  # narrow retrieval
    {"k": 4, "max_tokens": 300, "temperature": 0.7, "top_p": 0.90, "top_k": 50},  # creative / longer
    {"k": 5, "max_tokens": 350, "temperature": 0.2, "top_p": 0.95, "top_k": 40},  # detailed & focused
]

for i, (q, cfg) in enumerate(zip(questions, tuning_configs), start=1):
    print(f"--- Combination {i} | k={cfg['k']} temperature={cfg['temperature']} "
          f"top_p={cfg['top_p']} top_k={cfg['top_k']} max_tokens={cfg['max_tokens']} ---")
    print("Q:", q, "\n")
    print(generate_rag_response(
        q, k=cfg["k"], max_tokens=cfg["max_tokens"],
        temperature=cfg["temperature"], top_p=cfg["top_p"], top_k=cfg["top_k"]))
    print("\n" + "=" * 100 + "\n")

--- Combination 1 | k=3 temperature=0.0 top_p=0.95 top_k=20 max_tokens=200 ---
Q: What is the protocol for managing sepsis in a critical care unit? 



Llama.generate: prefix-match hit


###Answer
The protocol for managing sepsis in a critical care unit includes obtaining cultures of blood and any other appropriate specimens, administering empiric antibiotics after cultures are obtained, adjusting antibiotics according to culture and susceptibility testing results, surgically draining any abscesses, removing internal devices that may be the source of bacteria, and monitoring vital signs, fluid intake and output, temperature, blood pressure, pulse, respiration rate, electrolytes, CBC, Mg, phosphate, Ca levels for patients with arrhythmias, and liver enzymes and coagulation profiles for patients receiving TPN. Point-of-care testing may be used to do certain blood tests at the patient's bedside or unit. Other emerging therapies include cooling for hyperthermia and early treatment of renal failure with continuous venovenous hemofiltration.


--- Combination 2 | k=5 temperature=0.3 top_p=0.95 top_k=50 max_tokens=256 ---
Q: What are the common symptoms for appendicitis, and 

Llama.generate: prefix-match hit


Based on the provided context, sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder affecting genetically susceptible individuals exposed to unclear environmental triggers. The treatment options for addressing this condition include:

1. Topical corticosteroids: These medications can stimulate hair growth and are usually effective within 6 to 8 months of treatment. However, they may cause adverse effects such as decreased libido, erectile and ejaculatory dysfunction, hypersensitivity reactions, gynecomastia, and myopathy. Long-term use is common due to the positive results.
2. Minoxidil: A topical medication that can promote hair growth in affected areas.
3. Anthralin: Another topical treatment for alopecia areata.
4. Immunotherapy (diphencyprone or squaric acid dibutylester): These agents work by inducing an allergic reaction to stimulate hair regrowth.
5. Psoralen plus ultraviolet A (PUVA): This is a systemic treatment that involves the use of oral psoral

Llama.generate: prefix-match hit


###Answer
The context suggests that there is no specific treatment for brain tissue injury other than supportive care. This includes preventing systemic complications, providing good nutrition, and preventing pressure ulcers. Rehabilitation is needed when neurologic deficits persist, which may include physical, occupational, and speech therapy, skill-building activities, and counseling. Emotional support is also crucial for patients and their families. Treatments to promote nerve regeneration, such as injections of autologous macrophages or epidural administration of BA-210, are under study. Stem cell research shows promising results but is still in its infancy. No mention was found regarding surgical interventions specifically for brain tissue injury in the context.


--- Combination 5 | k=5 temperature=0.2 top_p=0.95 top_k=40 max_tokens=350 ---
Q: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be c

#### Observations - Question Answering using RAG (and fine-tuning)

- With retrieval, each answer is grounded in the **top-k chunks from the Merck Manual**, so responses become **more specific and source-faithful**, and hallucination drops noticeably versus the base model.
- Fine-tuning across five combinations of **retriever `k`** and **LLM parameters** exposes the trade-off: small `k` (2-3) keeps the context tight and on-topic; larger `k` (4-5) supplies more evidence but can dilute the prompt and risk exceeding `n_ctx`.
- **Chunking** matters too: a smaller `chunk_size` improves retrieval precision but fragments context; larger chunks preserve context but retrieve more noise. (Changing `chunk_size`/`chunk_overlap` requires rebuilding the vector store.)
- A low `temperature` with `k=3-4` and `max_tokens ~= 256` was generally a good balance of accuracy and completeness.

_Confirm with your run and record the best configuration per question._

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [33]:
groundedness_rater_system_message = """
You are tasked with rating AI-generated answers to questions posed by users.
You will be presented a QUESTION, the CONTEXT that was used by the AI system to generate the answer, and the ANSWER itself.

Your task is to evaluate GROUNDEDNESS: how well the answer is supported by the given context.

Rate the answer on an integer scale from 1 to 5, where:
1 - the answer is not supported by the context at all (fully hallucinated).
5 - every claim in the answer is directly supported by the context.

Do NOT judge the answer against your own knowledge; only judge whether it is grounded in the provided context.
Provide the rating as a single integer, followed by a brief justification.
"""

In [34]:
relevance_rater_system_message = """
You are tasked with rating AI-generated answers to questions posed by users.
You will be presented a QUESTION, the CONTEXT that was used by the AI system to generate the answer, and the ANSWER itself.

Your task is to evaluate RELEVANCE: how well the answer addresses the user's question.

Rate the answer on an integer scale from 1 to 5, where:
1 - the answer is completely irrelevant to the question.
5 - the answer fully and directly addresses the question.

Provide the rating as a single integer, followed by a brief justification.
"""

In [35]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [39]:
def generate_ground_relevance_response(user_input,k=4,max_tokens=256,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [40]:
ground, rel = generate_ground_relevance_response(user_input="What is the protocol for managing sepsis in a critical care unit?", max_tokens=370)

print(ground, end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer a 5. The context provides detailed information about the care provided in an ICU, including monitoring, blood tests, and treatment options for various conditions. The answer accurately summarizes this information as it relates to managing sepsis in a critical care unit, including obtaining cultures and blood tests, administering antibiotics, and providing supportive care.

 I would rate the answer a 5. The AI-generated answer directly addresses the user's question by providing a detailed protocol for managing sepsis in a critical care unit, including obtaining cultures, administering antibiotics, supportive care, continuous monitoring, and other emerging therapies.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [41]:
ground, rel = generate_ground_relevance_response(user_input="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?", max_tokens=370)

print(ground, end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer as a 5. The context provided in the text thoroughly covers the symptoms of appendicitis and the fact that it requires surgical removal for treatment. The answer accurately reflects this information, making every claim directly supported by the context.

 I would rate the answer as 5. The AI-generated answer fully and directly addresses the user's question by providing a detailed explanation of the common symptoms for appendicitis, stating that it cannot be cured via medicine alone, and describing the surgical procedure (appendectomy) that should be followed to treat it. The answer also mentions the use of IV antibiotics before surgery and provides additional information about what to do in cases where a pericolic abscess has already formed or a Meckel's diverticulum is present.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [42]:
ground, rel = generate_ground_relevance_response(user_input="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?", max_tokens=370)

print(ground, end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer as 5. The context clearly mentions alopecia areata as a cause of sudden patchy hair loss and provides several effective treatments for it, including topical and intralesional corticosteroids, minoxidil, anthralin, immunotherapy, hormonal modulators, and systemic corticosteroids. The answer accurately reflects the information in the context and also mentions possible causes other than alopecia areata, such as infections and medications.

 I would rate the answer as 5. The answer fully addresses the user's question by providing information on the causes of sudden patchy hair loss (alopecia areata) and effective treatments for this condition, including topical and systemic options. The answer also mentions other possible causes of sudden patchy hair loss, such as infections and medications. Overall, the answer provides a comprehensive response to the user's question.


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [43]:
ground, rel = generate_ground_relevance_response(user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?", max_tokens=370)

print(ground, end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer a 5. The context clearly states that supportive care is recommended for individuals with brain injuries, including preventing systemic complications, providing good nutrition, and preventing pressure ulcers. It also mentions the need for rehabilitation when neurologic deficits persist, which includes physical, occupational, and speech therapy, skill-building activities, and counseling. The answer accurately reflects this information from the context.

 I would rate the answer a 5. The answer directly addresses the user's question by summarizing the recommended treatments for a person with a physical injury to brain tissue, which includes supportive care and rehabilitation. The answer also provides important information about the prognosis for recovery in cases of persistent vegetative state.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [44]:
ground, rel = generate_ground_relevance_response(user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?", max_tokens=370)

print(ground, end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 5: The answer is thoroughly grounded in the context provided. The context discusses various precautions and treatments for injuries, including immobilization, pain management, definitive treatment, and RICE therapy. The answer summarizes these points specifically for a person who has fractured their leg during a hiking trip.

 5 - The answer fully and directly addresses the user's question by providing a comprehensive list of necessary precautions and treatment steps for someone who has fractured their leg during a hiking trip, including immobilization, pain management, definitive treatment, RICE therapy, and rehabilitation considerations.


#### Observations - Output Evaluation (LLM-as-a-judge)

- Each RAG answer is scored on **groundedness** (is it supported by the retrieved context?) and **relevance** (does it answer the question?), using the same Mistral model as the judge.
- Well-retrieved questions (e.g., sepsis, appendicitis) should score **high on both**. A question whose answer is only thinly covered by the retrieved chunks may score lower on groundedness - a cue to raise `k` or adjust chunking.
- LLM-as-judge is a useful, low-cost proxy but is **not infallible** (the model is rating itself); for production, complement it with expert human review.

_Replace with the actual scores and short justifications your run produces._

## Actionable Insights and Business Recommendations

Based on this prototype:

- **RAG grounded in the Merck Manual substantially improves answer reliability** over a base LLM - answers become specific, source-backed, and far less prone to hallucination, which is essential in a clinical setting.
- **Prompt engineering and parameter tuning** (system role, temperature, `k`, `max_tokens`) are low-effort levers that materially change answer quality; a conservative setting (low temperature, `k=3-4`, sufficient `max_tokens`) is a sensible default for medical use.
- **A groundedness + relevance evaluation loop** should be built into the pipeline so answer quality is monitored continuously rather than assumed.
- **Business value:** faster access to trusted knowledge at the point of care, more standardized guidance across clinicians, and far less time spent searching 4,000+ pages of manuals - supporting better and quicker decisions, especially in emergencies.
- **Recommended next steps:** keep a clinician in the loop for verification, expand the knowledge base with additional trusted manuals/guidelines, add source-page citations to each answer, and monitor for hallucinations and out-of-context questions before any real deployment.

_These are starting points - tailor them to the specific findings from your run._

In [45]:
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/Full_Code_NLP_RAG_Ravikanth_Doddapaneni_V12.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/Full_Code_NLP_RAG_Ravikanth_Doddapaneni_V12.ipynb to html
[NbConvertApp] Writing 639960 bytes to /content/drive/MyDrive/Colab Notebooks/Full_Code_NLP_RAG_Ravikanth_Doddapaneni_V12.html


<font size=6 color='blue'>Power Ahead</font>
___